# Step 1

## Парсятся страны с кодами и загружаются в БД

In [19]:
from datetime import datetime
from os import getenv

import pandas as pd
from sqlalchemy import create_engine, text


In [5]:
engine = create_engine(getenv("SQLALCHEMY_RELOHELPER_URL"))

# I get it from Wikipedia and save country codes and the U.S. state codes tables in the database.
And make those codes the primary keys.

### Table of all countries and Alpha 3 codes

In [6]:
url_countries = 'https://en.wikipedia.org/wiki/ISO_3166-1'

In [7]:
df_countries = pd.read_html(
    url_countries, match="Alpha-3 code", storage_options={"User-Agent": "Mozilla/5.0"}
)[0]

df_countries = df_countries.rename(
    columns={
        "English short name (using title case)": "country",
        "Alpha-2 code": "alpha2_code",
        "Alpha-3 code": "country_code",
        "Numeric code": "numeric_code",
        "Independent[b]": "independent",
    }
)

df_countries = df_countries.drop(columns=["Link to ISO 3166-2"])

In [8]:
df_countries

,country,alpha2_code,country_code,numeric_code,independent
0,Afghanistan[c],AF,AFG,4,Yes
1,Åland Islands,AX,ALA,248,No
2,Albania,AL,ALB,8,Yes
3,Algeria,DZ,DZA,12,Yes
4,American Samoa,AS,ASM,16,No
...,...,...,...,...,...
244,Wallis and Futuna,WF,WLF,876,No
245,Western Sahara[c],EH,ESH,732,No
246,Yemen,YE,YEM,887,Yes
247,Zambia,ZM,ZMB,894,Yes


Делаем очистку названия страны:
- `\[.*?\]` — находим любой текст в квадратных скобках ([c], [note 1], и т.д.)
- удаляем его
- `strip()` убираем лишние пробелы

In [9]:
for country_code, row in df_countries.iterrows():
    if country_code != 'VGB' and country_code != 'VIR':
        df_countries["country"] = (
            df_countries["country"].str.replace(r"\[.*?\]", "", regex=True).str.strip()
        )

In [10]:
df_countries.head(2)

,country,alpha2_code,country_code,numeric_code,independent
0,Afghanistan,AF,AFG,4,Yes
1,Åland Islands,AX,ALA,248,No


# Total Population by Country

Downloaded from https://worldpopulationreview.com/countries

In [ ]:
# Читаем и обрабатываем df_population
df_population = pd.read_csv("./data/total-population-by-country-2026.csv", sep=",")
df_population.rename(
    columns={"cca3": "country_code", "pop2026": "population"}, inplace=True
)

In [ ]:
# Приводим колонку area к int (предполагаю, что вы имели в виду area, а не aria)
# Сначала удаляем NaN, потом конвертируем
df_population["area"] = (
    pd.to_numeric(df_population["area"], errors="coerce").fillna(0).astype(int)
)

In [ ]:
df_population.head(3)

,population,pop2050,country,area,landAreaKm,cca2,country_code,density,growthRate,worldPercentage,rank
0,1476630000,1679590000,India,3287590,2973190.0,IN,IND,496.6484,0.0087,0.1845,1
1,1412910000,1260290000,China,9706961,9424702.9,CN,CHN,149.9156,-0.0023,0.1765,2
2,349035000,380847000,United States,9372610,9147420.0,US,USA,38.1567,0.0051,0.0436,3


In [ ]:
# Делаем merge (join) по колонке 'country_code'
df_result = df_countries.merge(
    df_population[
        ["country_code", "population", "area"]
    ],  # берем только нужные колонки из df_population
    on="country_code",  # ключ для соединения
    how="left",  # left join чтобы сохранить все страны из df_countries
)

# Конвертируем в int, но сначала заполняем NaN
df_result["population"] = df_result["population"].fillna(0).astype(int)
df_result["area"] = df_result["area"].fillna(0).astype(int)

# Добавляем колонку с текущей датой
df_result["last_update"] = datetime.now().strftime("%Y-%m-%d")

In [ ]:
df_result.head(3)

,country,alpha2_code,country_code,numeric_code,independent,population,area,last_update
0,Afghanistan,AF,AFG,4,Yes,45047100,652230,2026-03-17
1,Åland Islands,AX,ALA,248,No,0,0,2026-03-17
2,Albania,AL,ALB,8,Yes,2751020,28748,2026-03-17


## Load df to DB

In [31]:
# Проверка
assert df_result["country_code"].notna().all()
assert df_result["country_code"].is_unique

In [ ]:
df_result.to_sql(
    "countries",
    engine,
    index=False,
    if_exists="fail",
)

with engine.connect() as connection:
    connection.execute(
        text(
            "ALTER TABLE countries "
            "ADD CONSTRAINT pk_country_code PRIMARY KEY (country_code)"
        )
    )
    connection.commit()


# Table of all U.S. states in ISO format

In [ ]:
url_states = 'https://en.wikipedia.org/wiki/ISO_3166-2:US'
df_states = pd.read_html(url_states, storage_options={"User-Agent": "Mozilla/5.0"})[0]
df_states.columns = ['state_code', 'state_name', 'category']
df_states.head(2)

In [ ]:
df_states.to_sql('states', engine, index=False)
with engine.connect() as connection:
    connection.execute(text("ALTER TABLE states ADD CONSTRAINT PK_state_code PRIMARY KEY (state_code)"))
    connection.commit()